In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 74.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6545c666af18cdc283093162fa7950a5ce3a0b78ad620ee3230dd244fca59a09
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [6]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# ─────────────────────────────────────────────────────────────
# BB84 Quantum Key Distribution — No Attacker
# Agents: Alice (sender) | Bob (receiver)
# ─────────────────────────────────────────────────────────────

simulator = BasicSimulator()

# ── Quantum Random Number Generator ───────────────────────────────────────────
# Prepares n qubits in |+⟩ = (|0⟩+|1⟩)/√2 and measures.
# Required by assignment — do NOT use Python's random module.

def quantum_random_bits(n):
    bits = []
    batch = 20
    while len(bits) < n:
        size = min(batch, n - len(bits))
        qc = QuantumCircuit(size, size)
        qc.h(range(size))
        qc.measure(range(size), range(size))
        job = simulator.run(transpile(qc, simulator), shots=1)
        result = list(job.result().get_counts().keys())[0]
        bits += [int(b) for b in reversed(result)]
    return bits[:n]


# ══════════════════════════════════════════════════════════════
# ALICE — prepares and sends qubits
# ══════════════════════════════════════════════════════════════

def alice_prepare(n):
    """
    Alice generates:
      - n random bits  (the raw key material)
      - n random bases (0 = rectilinear {|0⟩,|1⟩}, 1 = diagonal {|+⟩,|−⟩})
    She encodes each bit into a single-qubit circuit:
      bit=0, basis=0 → |0⟩        (do nothing)
      bit=1, basis=0 → |1⟩        (X gate)
      bit=0, basis=1 → |+⟩        (H gate)
      bit=1, basis=1 → |−⟩        (X then H)
    """
    bits  = quantum_random_bits(n)
    bases = quantum_random_bits(n)

    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)       # encode bit value
        if basis == 1:
            qc.h(0)       # rotate to diagonal basis
        circuits.append(qc)

    print("[Alice] raw bits:  ", bits)
    print("[Alice] bases:     ", bases, "  (0=rectilinear, 1=diagonal)")
    return circuits, bits, bases


# ══════════════════════════════════════════════════════════════
# BOB — receives and measures qubits
# ══════════════════════════════════════════════════════════════

def bob_measure(alice_circuits, n):
    """
    Bob independently chooses n random bases and measures each qubit.
    When he picks basis=1 (diagonal), he applies H before measuring
    to rotate back into the computational basis.
    """
    bases   = quantum_random_bits(n)
    results = []

    for qc, basis in zip(alice_circuits, bases):
        # Build a fresh circuit: copy Alice's encoding, add Bob's measurement
        full_qc = QuantumCircuit(1, 1)
        full_qc.compose(qc, inplace=True)
        if basis == 1:
            full_qc.h(0)      # rotate from diagonal basis before measuring
        full_qc.measure(0, 0)

        job = simulator.run(transpile(full_qc, simulator), shots=1)
        bit = int(list(job.result().get_counts().keys())[0])
        results.append(bit)

    print("[Bob]   bases:     ", bases)
    print("[Bob]   results:   ", results)
    return results, bases


# ══════════════════════════════════════════════════════════════
# SIFTING — classical public channel
# ══════════════════════════════════════════════════════════════

def sift_key(alice_bits, alice_bases, bob_results, bob_bases):
    """
    Alice and Bob announce their bases publicly (NOT their bits).
    They discard every position where their bases differed —
    only matching-basis measurements are guaranteed to agree.
    Expect ~50% of bits to survive sifting.
    """
    alice_key, bob_key, kept = [], [], []

    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_key.append(alice_bits[i])
            bob_key.append(bob_results[i])
            kept.append(i)

    pct = len(alice_key) / len(alice_bits) * 100
    print(f"\n[Sift]  Kept positions: {kept}")
    print(f"[Sift]  Sifted key length: {len(alice_key)}/{len(alice_bits)} ({pct:.0f}%)")
    print("[Alice sifted key]", alice_key)
    print("[Bob   sifted key]", bob_key)
    return alice_key, bob_key


# ══════════════════════════════════════════════════════════════
# ERROR RATE CHECK — attack detection
# ══════════════════════════════════════════════════════════════

def check_error_rate(alice_key, bob_key, sample_fraction=0.2, threshold=0.11):
    """
    Alice and Bob sacrifice a random subset of their sifted key bits
    by revealing them over the classical channel to estimate the
    Quantum Bit Error Rate (QBER).

    In an ideal noiseless simulation with no attacker, QBER = 0.
    The standard security threshold is 11%: above this, an eavesdropper
    almost certainly caused the errors and the key is discarded.
    """
    import math
    n_sample = max(1, math.ceil(len(alice_key) * sample_fraction))
    # Use quantum RNG to pick sample indices
    sample_bits = quantum_random_bits(len(alice_key))
    indices = [i for i, b in enumerate(sample_bits) if b == 1][:n_sample]
    if not indices:
        indices = [0]

    errors = sum(alice_key[i] != bob_key[i] for i in indices)
    qber   = errors / len(indices)

    print(f"\n[Check] Sample size: {len(indices)}, Errors found: {errors}")
    print(f"[Check] QBER: {qber:.2%}  (threshold: {threshold:.0%})")

    if qber > threshold:
        print("[Check]  QBER exceeds threshold — possible eavesdropper! Key discarded.")
        return qber, True
    else:
        print("[Check] QBER within safe range — no eavesdropper detected.")
        return qber, False


# ══════════════════════════════════════════════════════════════
# RUN THE PROTOCOL
# ══════════════════════════════════════════════════════════════

N = 100   # number of qubits Alice sends

print("=" * 56)
print("BB84 — Plain (no attacker)")
print("=" * 56)

# Step 1 — Alice encodes
circuits, alice_bits, alice_bases = alice_prepare(N)

# Step 2 — Bob measures
print()
bob_results, bob_bases = bob_measure(circuits, N)

# Step 3 — Sift on classical channel
alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_results, bob_bases)

# Step 4 — Check error rate
qber, attack_detected = check_error_rate(alice_key, bob_key)

# Step 5 — Final result
print("\n── Final Result " + "─" * 40)
if not attack_detected:
    # Remove the sacrificed sample bits from the final key (simplified here)
    print(f"Shared secret key ({len(alice_key)} bits): {alice_key}")
    match = alice_key == bob_key
    print(f"Alice and Bob's keys match: {'Yes' if match else 'No'}")
else:
    print("Protocol aborted — keys discarded.")


BB84 — Plain (no attacker)
[Alice] raw bits:   [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0]
[Alice] bases:      [1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1]   (0=rectilinear, 1=diagonal)

[Bob]   bases:      [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 